## 工具包、公共函数、全局变量

In [2]:
import sys
# 载入的工具包
import numpy as np
import pandas as pd
import binascii
import datetime
import math
import re

# import globalVar as glv
class glv:
    def _init():
        global _global_dict
        _global_dict = {}


    def set_value(key,value):
        _global_dict[key] = value


    def get_value(key,defValue=None):
        try:
            return _global_dict[key]
        except KeyError:
            return defValue

# 公共函数
# 8进制转10进制
def oct2dec(x):
    output = int(str(x), 8)
    return output

# 16进制转10进制（先乘精度，后偏移）
def hex2dec(x, n=0, k=1, e=False):
    x = x.upper()
    k_n = 0 if k>=1 else int(math.log10(k))
    if e:
        if x == "EF" or x == "FFFE" or x == "FFFFFFFE":
            return "异常"
        elif x == "FF" or x == "FFFF" or x == "FFFFFFFF":
            return "无效"
        else:
            output_s = int(str(x), 16)
            output = output_s*k + n
            return round(output, -k_n)
    else :
        output_s = int(str(x), 16)
        output = output_s*k + n
        return round(output, -k_n)    

# 16进制转字符串
def hex2str(x):
    output = binascii.a2b_hex(x).decode("utf8")
    return output

# 时间
def get_datetime(x):
    year = hex2dec(x[0:2]) + 2000
    month = hex2dec(x[2:4])
    day = hex2dec(x[4:6])
    hour = hex2dec(x[6:8])
    minute = hex2dec(x[8:10])
    second = hex2dec(x[10:12])
    output = datetime.datetime(year, month, day, hour, minute, second).strftime('%Y-%m-%d %H:%M:%S')
    return output

# 构建
def list2dict(data, list_o, cf_a, mode='s'):
    dict_o = {}
    if mode == 's':
        for i in range(len(cf_a)-1):
            dict_o[list_o[i]] = data[cf_a[i]:cf_a[i+1]]
        return dict_o
    elif mode == 'l':
        for i in range(len(cf_a)-1):
            dict_o[list_o[i]] = [data[cf_a[i]:cf_a[i+1]]]
        return dict_o

# 累加
def hexlist2(data, n=1):
    output = [0]
    data = data * n
    for i in range(0, len(data)):
        output.append(output[i] + data[i])
    output = [i*2 for i in output]
    return output

# 列表
def hex2list(data, num=1, kn=0, kk=1, ke=False):
    num = num * 2
    n = '.{'+str(num)+'}'
    output = re.findall(n, data) 
    output = [hex2dec(i, n=kn ,k=kk ,e=ke) for i in output]
    return output
# 报文解析
# 解析替换函数
def dict_list_replace(n, x):
    x = x.upper()
    try:
        index = jx_dict_o[n].index(x)
        output = jx_dict[n][index]
    except:
        output = "ERROR"
    return output

# 解析列表
jx_dict_o = {
    "02":["01", "02", "03", "04", "05", "06"],
    "03":["01", "02", "03", "FE"],
    "05":["01", "02", "03", "FE", "FF"],
    "07_02_01_01":["01", "02", "03", "FE", "FF"],
    "07_02_01_02":["01", "02", "03", "04", "FE", "FF"],
    "07_02_01_03":["01", "02", "03", "FE", "FF"],
    "07_02_01_06":["01", "02", "FE", "FF"],
    "07_02_02_02":["01", "02", "03", "04", "FE", "FF"],
    "07_02_03_12":["01", "02", "FE", "FF"],
    "07_02_04_01":["01", "02", "FE", "FF"],
    "07_05_05":["01", "02", "03", "FE", "FF"],
}

jx_dict = {
    "02":["车辆登入", "实时信息上报", "补发信息上报", "车辆登出", "平台登入", "平台登出"],
    "03":["成功", "错误", "VIN重复", "命令"],
    "05":["不加密", "RSA加密", "AES128位加密", "异常", "无效"],
    "07_02_01_01":["启动", "熄火", "其他", "异常", "无效"],
    "07_02_01_02":["停车充电", "行驶充电", "未充电", "充电完成", "异常", "无效"],
    "07_02_01_03":["纯电", "混动", "燃油", "异常", "无效"],
    "07_02_01_06":["工作", "断开", "异常", "无效"],
    "07_02_02_02":["耗电", "发电", "关闭", "准备", "异常", "无效"],
    "07_02_03_12":["工作", "断开", "异常", "无效"],
    "07_02_04_01":["启动", "关闭", "异常", "无效"],
    "07_05_05":["不加密", "RSA加密", "AES128位加密", "异常", "无效"],
}

## fun_01to06

In [299]:
class fun_01to06(object):
    def __init__(self, data):
        self.cf = [2, 1, 1, 17, 1, 2]
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "起始符",
            "命令标识",
            "应答标志",
            "唯一识别码",
            "数据单元加密方式",
            "数据单元长度"
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "起始符":hex2str(self.oj["起始符"]),
            "命令标识":dict_list_replace('02', self.oj['命令标识']),
            "应答标志":dict_list_replace('03', self.oj['应答标志']),
            "唯一识别码":hex2str(self.oj["唯一识别码"]),
            "数据单元加密方式":dict_list_replace('05', self.oj['数据单元加密方式']),
            "数据单元长度":hex2dec(self.oj["数据单元长度"]),
        }
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        self.mo = self.oj["命令标识"]
        
        glv.set_value('data_f', self.next)
        glv.set_value('data_mo', self.mo)
        glv.set_value('data_01to07', self.o)

In [300]:
glv.get_value("data_mo")

'02'

## fun_07

In [301]:
class fun_07:
    def __init__(self, data):
        
        self.mo = glv.get_value("data_mo")
        if self.mo == '01':
            self.o = fun_07_01(glv.get_value('data_f'))
        elif self.mo == '02' or self.mo == '03':
            self.o = fun_07_02(glv.get_value('data_f'))
        elif self.mo == '04':
            self.o = fun_07_04(glv.get_value('data_f'))
        elif self.mo == '05':
            self.o = fun_07_05(glv.get_value('data_f'))
        elif self.mo == '06':
            self.o = fun_07_06(glv.get_value('data_f'))
        else :
            print('命令标识:',self.mo,'有误')
        
        self.c = fun_07_cursor(glv.get_value('data_f'))
        
        self.oj = dict(self.o.oj, **self.c.oj)
        self.oj2 = {'数据单元':self.oj}
        self.ol = pd.merge(self.o.ol, self.c.ol, left_index=True, right_index=True)
        self.pj = dict(self.o.pj, **self.c.pj)
        self.pj2 = {'数据单元':self.pj}
        self.pl = pd.merge(self.o.pl, self.c.pl, left_index=True, right_index=True)
        
        print('fun_07 done!')

## fun_07_01

In [302]:
class fun_07_01(object):
    def __init__(self, data):
        self.cf = [6, 2, 20, 1, 1]
        self.cf_a = hexlist2(self.cf)
        self.n = hex2dec(data[self.cf_a[3]:self.cf_a[4]])
        self.m = hex2dec(data[self.cf_a[4]:self.cf_a[5]])
        self.cf.append(self.n*self.m)
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "数据采集时间",
            "登入流水号",
            "ICCID",
            "可充电储能子系统数",
            "可充电储能系统编码长度",
            "可充电储能系统编码",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'车辆登入': self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "数据采集时间":get_datetime(self.oj['数据采集时间']),
            "登入流水号":hex2dec(self.oj['登入流水号']),
            "ICCID":hex2str(self.oj['ICCID']),
            "可充电储能子系统数":hex2dec(self.oj['可充电储能子系统数']),
            "可充电储能系统编码长度":hex2dec(self.oj['可充电储能系统编码长度']),
            "可充电储能系统编码":fun_07_01.fun_07_01_06(self.oj['可充电储能系统编码'], self.oj['可充电储能子系统数'], self.oj['可充电储能系统编码长度']),
        }
        self.pj2 = {'车辆登入': self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        
        glv.set_value('data_07_01', self.o)
        
        print('fun_07_01 done!')
        print(self.o)
        
    def fun_07_01_06(data, n, m):
        if m=='00':
            return "NA"
        else :
            n = hex2dec(n)
            m = hex2dec(m) * 2
            output = []
            for i in range(n):                
                output_unit = hex2str(data[i * m: i* m +m])
                output.append(output_unit)
            return output

## fun_07_04

In [303]:
class fun_07_04(object):
    def __init__(self, data):
        self.cf = [6, 2]
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "登出时间",
            "登出流水号",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "登出时间":get_datetime(self.oj['登出时间']),
            "登出流水号":hex2dec(self.oj['登出流水号']),
        }
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        
        glv.set_value('data_07_04', self.o)
        
        print('fun_07_04 done!')
        print(self.o)
        

## fun_07_05

In [304]:
class fun_07_05(object):
    def __init__(self, data):
        self.cf = [6, 2, 12, 20, 1]
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "平台登入时间",
            "登入流水号",
            "平台用户名",
            "平台密码",
            "加密规则",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "平台登入时间":get_datetime(self.oj['平台登入时间']),
            "登入流水号":hex2dec(self.oj['登入流水号']),
            "平台用户名":hex2str(self.oj['平台用户名']),
            "平台密码":hex2str(self.oj['平台密码']),
            "加密规则":dict_list_replace('07_05_05',self.oj['加密规则']),
        }
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('data_07_05', self.o)
        print('fun_07_05 done!')
        print(self.o)
        

## fun_07_06

In [305]:
class fun_07_07(object):
    def __init__(self, data):
        self.cf = [6, 2]
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "登出时间",
            "登出流水号",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        print(self.oj)
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "登出时间":get_datetime(self.oj['登出时间']),
            "登出流水号":hex2dec(self.oj['登出流水号']),
        }
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('data_07_06', self.o)
        print('fun_07_06 done!')
        print(self.o)
        

## fun_07_02

In [306]:
class fun_07_02:
    def __init__(self, data):
        self.o = data
        
        self.oj = {'数据采集时间': self.o[:12]}
        self.ol = pd.DataFrame({'01':['01']})
        self.pj = {'数据采集时间': get_datetime(self.oj['数据采集时间'])}
        self.pl = pd.DataFrame({'01':['01']})
        
        glv.set_value('data_f', data[12:])
        glv.set_value('m_07_02', data[12:14])
        
#         self.mo_list = glv.get_value('data_mo_list')
        self.mo_list = ['01','02','05','06','07','08','09']
        self.do_list = []
        
        while(glv.get_value('m_07_02') in self.mo_list):
            
            # 记录已执行的
            self.do_list.append(glv.get_value('m_07_02'))
            # 删除已执行的
            self.mo_list.remove(glv.get_value('m_07_02'))
            
            if glv.get_value('m_07_02') == '01':
                self.f_01 = fun_07_02_01(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '02':
                self.f_02 = fun_07_02_02(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '03':
                self.f_03 = fun_07_02_03(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '04':
                self.f_04 = fun_07_02_04(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '05':
                self.f_05 = fun_07_02_05(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '06':
                self.f_06 = fun_07_02_06(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '07':
                self.f_07 = fun_07_02_07(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '08':
                self.f_08 = fun_07_02_08(glv.get_value('data_f'))
            elif glv.get_value('m_07_02') == '09':
                self.f_09 = fun_07_02_09(glv.get_value('data_f'))
            else:
                print("fun_07_02 done")
                print(glv.get_value('data_f'))
                print(glv.get_value('m_07_02'))
            
        self.do_list.sort()
        for i in self.do_list:
            if i == '01':
                self.oj = dict(self.oj,**self.f_01.oj2)
                self.ol = pd.merge(self.ol, self.f_01.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_01.pj2)
                self.pl = pd.merge(self.pl, self.f_01.pl, left_index=True, right_index=True)
            elif i == '02':
                self.oj = dict(self.oj,**self.f_02.oj2)
                self.ol = pd.merge(self.ol, self.f_02.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_02.pj2)
                self.pl = pd.merge(self.pl, self.f_02.pl, left_index=True, right_index=True)
            elif i == '03':
                self.oj = dict(self.oj,**self.f_03.oj2)
                self.ol = pd.merge(self.ol, self.f_03.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_03.pj2)
                self.pl = pd.merge(self.pl, self.f_03.pl, left_index=True, right_index=True)
            elif i == '04':
                self.oj = dict(self.oj,**self.f_04.oj2)
                self.ol = pd.merge(self.ol, self.f_04.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_04.pj2)
                self.pl = pd.merge(self.pl, self.f_04.pl, left_index=True, right_index=True)
            elif i == '05':
                self.oj = dict(self.oj,**self.f_05.oj2)
                self.ol = pd.merge(self.ol, self.f_05.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_05.pj2)
                self.pl = pd.merge(self.pl, self.f_05.pl, left_index=True, right_index=True)
            elif i == '06':
                self.oj = dict(self.oj,**self.f_06.oj2)
                self.ol = pd.merge(self.ol, self.f_06.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_06.pj2)
                self.pl = pd.merge(self.pl, self.f_06.pl, left_index=True, right_index=True)
            elif i == '07':
                self.oj = dict(self.oj,**self.f_07.oj2)
                self.ol = pd.merge(self.ol, self.f_07.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_07.pj2)
                self.pl = pd.merge(self.pl, self.f_07.pl, left_index=True, right_index=True)
            elif i == '08':
                self.oj = dict(self.oj,**self.f_08.oj2)
                self.ol = pd.merge(self.ol, self.f_08.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_08.pj2)
                self.pl = pd.merge(self.pl, self.f_08.pl, left_index=True, right_index=True)
            elif i == '09':
                self.oj = dict(self.oj,**self.f_09.oj2)
                self.ol = pd.merge(self.ol, self.f_09.ol, left_index=True, right_index=True)
                self.pj = dict(self.pj,**self.f_09.pj2)
                self.pl = pd.merge(self.pl, self.f_09.pl, left_index=True, right_index=True)
                
                
                
        self.oj2 = {'信息上报': self.oj}
        self.pj2 = {'信息上报': self.pj}

        print('fun_07_02 done!')

## fun_07_02_01

In [307]:
class fun_07_02_01(object):
    def __init__(self, data):
        self.cf = [1, 1, 1, 2, 4, 2, 2, 1, 1, 1, 2, 1, 1]
        self.cf_a = hexlist2(self.cf)
        data = data[2:]
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            '车辆状态',
            '充电状态',
            '运行模式',
            '车速',
            '累计里程',
            '总电压',
            '总电流',
            'SOC',
            'DC-DC状态',
            '挡位',
            '绝缘电阻',
            '加速踏板行程值',
            '制动踏板状态',
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'整车数据': self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            '车辆状态' : dict_list_replace("07_02_01_01", self.oj['车辆状态']),
            '充电状态' : dict_list_replace("07_02_01_02", self.oj['充电状态']),
            '运行模式' : dict_list_replace("07_02_01_03", self.oj['运行模式']),
            '车速' : hex2dec(self.oj['车速'], k=0.1),
            '累计里程' : hex2dec(self.oj['累计里程'], k=0.1),
            '总电压' : hex2dec(self.oj['总电压'], k=0.1),
            '总电流' : hex2dec(self.oj['总电流'], n=-1000, k=0.1),
            'SOC' : hex2dec(self.oj['SOC']),
            'DC-DC状态' : dict_list_replace("07_02_01_06", self.oj['DC-DC状态']),
            '挡位' : fun_07_02_01.fun_10(self.oj['挡位']),
            '绝缘电阻' : hex2dec(self.oj['绝缘电阻']),
            '加速踏板行程值' : fun_07_02_01.fun_12(self.oj['加速踏板行程值']),
            '制动踏板状态' : fun_07_02_01.fun_13(self.oj['制动踏板状态']),
        }
        self.pj2 = {'整车数据': self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        glv.set_value('data_07_02_01', self.o)
        print(self.next, 'fun0201 test data_f')
        
        print('fun_07_02_01 done!')
        print(self.o)
        
        
    # 02_01_10 挡位
    def fun_10(data):
        n = '{:08b}'.format(int(data, 16))
        dangwei = n[-4:]
        zhidongli = n[-5]
        qudongli = n[-6]
        
        # 挡位
        if dangwei == '0000':
            dangwei_s = '空挡'
        elif dangwei == '1101':
            dangwei_s = '倒挡'
        elif dangwei == '1110':
            dangwei_s = '自动D挡'
        elif dangwei == '1111':
            dangwei_s = '停车P挡'
        else :
            dangwei_s = (str(int(dangwei, 2)) + "档" )
            
        # 制动力
        if zhidongli == "1":
            zhidongli_s = "有制动力"
        else :
            zhidongli_s = "无制动力" 
            
        # 驱动力
        if qudongli == "1":
            qudongli_s = "有驱动力"
        else :
            qudongli_s = "无驱动力"

        output = [n, dangwei_s, zhidongli_s, qudongli_s]
        return output

    # 02_01_12 加速踏板行程值
    def fun_12(data):
        data = data.upper()
        if data == 'FE':
            return "异常"
        elif data == "FF":
            return "无效"
        else :
            return hex2dec(data)

    # 02_01_13 制动踏板状态
    def fun_13(data):
        data = data.upper()
        if data == 'FE':
            return "异常"
        elif data == "FF":
            return "无效"
        elif data == "65":
            return "制动有效"
        else :
            return hex2dec(data)

## fun_07_02_02

In [308]:
class fun_07_02_02(object):
    def __init__(self, data):
        data = data[2:]
        self.dj_n_o = data[0:2]
        self.dj_n_j = hex2dec(self.dj_n_o) # 电机个数
        
        self.cf_u = [1, 1, 1, 2, 2, 1, 2, 2]
        self.cf = [1] + self.cf_u * self.dj_n_j
        self.cf_a = hexlist2(self.cf)
        
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "驱动电机个数",
            "驱动电机序号",
            "驱动电机状态",
            "驱动电机控制器温度",
            "驱动电机转速",
            "驱动电机转矩",
            "驱动电机温度",
            "电机控制器输入电压",
            "电机控制器直流母线电流",
        ]
        
        self.oj = fun_07_02_02.fun_oj(self)
        self.oj2 = {'驱动电机数据':self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "驱动电机个数":self.dj_n_j,
            "驱动电机序号":[hex2dec(i) for i in self.oj['驱动电机序号']],
            "驱动电机状态":[dict_list_replace('07_02_02_02', i) for i in self.oj['驱动电机状态']],
            "驱动电机控制器温度":[hex2dec(i, n=-40) for i in self.oj['驱动电机控制器温度']],
            "驱动电机转速":[hex2dec(i, n=-20000) for i in self.oj['驱动电机转速']],
            "驱动电机转矩":[hex2dec(i, k=0.1, n=-2000) for i in self.oj['驱动电机转矩']],
            "驱动电机温度":[hex2dec(i, n=-40) for i in self.oj['驱动电机温度']],
            "电机控制器输入电压":[hex2dec(i, k=0.1) for i in self.oj['电机控制器输入电压']],
            "电机控制器直流母线电流":[hex2dec(i, k=0.1, n=-1000) for i in self.oj['电机控制器直流母线电流']],
        }
        self.pj2 = {'驱动电机数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_02', self.o)
        
        print("fun_07_02_02 done!")
        print(self.o)
        
    #  oj
    def fun_oj(self):
        data = self.o[2:]
        cf_a = hexlist2(self.cf[1:])
        dict_oj = {
            "驱动电机个数":self.dj_n_o,
        }
        
        list_o = [
            "驱动电机序号",
            "驱动电机状态",
            "驱动电机控制器温度",
            "驱动电机转速",
            "驱动电机转矩",
            "驱动电机温度",
            "电机控制器输入电压",
            "电机控制器直流母线电流",
        ]
        
        dict_oj_u = {
            "驱动电机序号":[],
            "驱动电机状态":[],
            "驱动电机控制器温度":[],
            "驱动电机转速":[],
            "驱动电机转矩":[],
            "驱动电机温度":[],
            "电机控制器输入电压":[],
            "电机控制器直流母线电流":[],
        }
        for i in range(self.dj_n_j):
            for j in range(len(dict_oj_u)):
                data_unit = data[cf_a[i * len(dict_oj_u) + j]:cf_a[i * len(dict_oj_u) + j +1]]
                dict_oj_u[list_o[j]].append(data_unit)
        
        dict_all = dict(dict_oj, **dict_oj_u)
        return dict_all
    

## fun_07_02_03

In [309]:
class fun_07_02_03(object):
    def __init__(self, data):
        
        self.cf = [2, 2, 2, 2, None, 2, 1, 2, 1, 2, 1, 1]
        data = data[2:]
        self.dc_no = data[12:16]
        self.dc_np = hex2dec(self.dc_no)
        self.cf[4] = self.dc_np
        self.cf_a = hexlist2(self.cf)

        self.o = data[0:self.cf_a[-1]]
        
        self.list_o = [
            "燃料电池电压",
            "燃料电池电流",
            "燃料消耗率",
            "燃料电池温度探针总数",
            "探针温度值",
            "氢系统中最高温度",
            "氢系统中最高温度探针代号",
            "氢气最高浓度",
            "氢气最高浓度传感器代号",
            "氢气最高压力",
            "氢气最高压力传感器代号",
            "高压DC/DC状态",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'燃料电池数据':self.oj}
        self.ol = pd.DataFrame.from_dict(self.oj,orient='index').T
        self.pj = {
            "燃料电池电压":hex2dec(self.oj['燃料电池电压'], k=0.1),
            "燃料电池电流":hex2dec(self.oj['燃料电池电流'], k=0.1),
            "燃料消耗率":hex2dec(self.oj['燃料消耗率'], k=0.01),
            "燃料电池温度探针总数":hex2dec(self.oj['燃料电池温度探针总数']),
            "探针温度值":[hex2dec(i, n=-40, k=0.1) for i in self.oj['燃料电池温度探针总数']],
            "氢系统中最高温度":hex2dec(self.oj['氢系统中最高温度'], n=-40, k=0.1),
            "氢系统中最高温度探针代号":hex2dec(self.oj['氢系统中最高温度探针代号']),
            "氢气最高浓度":hex2dec(self.oj['氢气最高浓度']),
            "氢气最高浓度传感器代号":hex2dec(self.oj['氢气最高浓度传感器代号']),
            "氢气最高压力":hex2dec(self.oj['氢气最高压力'], k=0.1),
            "氢气最高压力传感器代号":hex2dec(self.oj['氢气最高压力传感器代号']),
            "高压DC/DC状态":dict_list_replace('07_02_03_12', self.oj['高压DC/DC状态']),
        }
        self.pj2 = {'燃料电池数据':self.pj}
        self.pl = pd.DataFrame.from_dict(self.pj,orient='index').T
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_03', self.o)
        
        print('fun_07_02_03 done!')
        print(self.o)
        

## fun_07_02_04

In [310]:
class fun_07_02_04(object):
    def __init__(self, data):
        cf = [1, 2, 2]
        cf_a = hexlist2(cf)
        data = data[2:]
        self.o = data[0:cf_a[-1]]
        list_o = [
            "发动机状态",
            "曲轴转速",
            "燃料消耗率",
        ]
        self.oj = list2dict(self.o, list_o, cf_a)
        self.oj2 = {'发动机数据':self.oj}
        self.ol = pd.DataFrame.from_dict(self.oj,orient='index').T
        self.pj = {
            "发动机状态":dict_list_replace("07_02_01_01", self.oj['发动机状态']),
            "曲轴转速":fun_07_02_04.fun_2(self.oj['曲轴转速']),
            "燃料消耗率":fun_07_02_04.fun_3(self.oj['燃料消耗率']),
        }
        self.pj2 = {'发动机数据':self.pj}
        self.pl = pd.DataFrame.from_dict(self.pj,orient='index').T
        
        self.next = data[cf_a[-1]:]
        self.nextMark = data[cf_a[-1]:cf_a[-1]+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_04', self.o)
        
        print('fun_07_02_04 done!')
        print(self.o)
        
    # 02_04_02 曲轴转速
    def fun_2(data):
        data = data.upper()
        if data == 'FFFE':
            return "异常"
        elif data == "FFFF":
            return "无效"
        else :
            return hex2dec(data)

    # 02_04_03 燃料消耗率
    def fun_3(data):
        data = data.upper()
        if data == 'FFFE':
            return "异常"
        elif data == "FFFF":
            return "无效"
        else :
            return hex2dec(data, k=0.01)

## fun_07_02_05

In [311]:
class fun_07_02_05(object):
    def __init__(self, data):
        self.cf = [1, 4, 4]
        self.cf_a = hexlist2(self.cf)
        data = data[2:]
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "定位状态",
            "经度",
            "纬度",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'车辆位置数据':self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            '定位状态' : fun_07_02_05.fun_01(self.oj['定位状态']),
            '经度' : hex2dec(self.oj['经度'], k=0.000001),
            '纬度' : hex2dec(self.oj['纬度'], k=0.000001),
        }
        self.pj2 = {'车辆位置数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_05', self.o)
        
        print('fun_07_02_05 done!')
        print(self.o)
        
    def fun_01(data):
        n = '{:08b}'.format(int(data, 16))    
        state = n[-1]
        longitude = n[-2]
        latitude = n[-3]
        
        if state == '0':
            state_s = "定位有效"
        else :
            state_s = "定位无效"
        
        if longitude == '0':
            longitude_s = "北纬"
        else :
            longitude_s = "南纬"
        
        if latitude == '0':
            latitude_s = "东经"
        else :
            latitude_s = "西经"
        
        output = [n, state_s, longitude_s, latitude_s]
        return output

## fun_07_02_06

In [312]:
class fun_07_02_06(object):
    def __init__(self, data):
        self.cf = [1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1]
        self.cf_a = hexlist2(self.cf)
        data = data[2:]
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "最高电压电池子系统号",
            "最高电压电池单体代号",
            "电池单体电压最高值",
            "最低电压电池子系统号",
            "最低电压电池单体代号",
            "电池单体电压最低值",
            "最高温度子系统号",
            "最高温度探针序号",
            "最高温度值",
            "最低温度子系统号",
            "最低温度探针序号",
            "最低温度值",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'极值数据':self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            '最高电压电池子系统号' : hex2dec(self.oj['最高电压电池子系统号'], e=True),
            '最高电压电池单体代号' : hex2dec(self.oj['最高电压电池单体代号'], e=True),
            '电池单体电压最高值' : hex2dec(self.oj['电池单体电压最高值'], k=0.001, e=True),
            '最低电压电池子系统号' : hex2dec(self.oj['最低电压电池子系统号'], e=True),
            '最低电压电池单体代号' : hex2dec(self.oj['最低电压电池单体代号'], e=True),
            '电池单体电压最低值' : hex2dec(self.oj['电池单体电压最低值'], k=0.001, e=True),
            '最高温度子系统号' : hex2dec(self.oj['最高温度子系统号'], e=True),
            '最高温度探针序号' : hex2dec(self.oj['最高温度探针序号'], e=True),
            '最高温度值' : hex2dec(self.oj['最高温度值'], n=-40, e=True),
            '最低温度子系统号' : hex2dec(self.oj['最低温度子系统号'], e=True),
            '最低温度探针序号' : hex2dec(self.oj['最低温度探针序号'], e=True),
            '最低温度值' : hex2dec(self.oj['最低温度值'], n=-40, e=True),
        }
        self.pj2 = {'极值数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_06', self.o)
        
        print('fun_07_02_06 done!')
        print(self.o)
        

## fun_07_02_07

In [313]:
class fun_07_02_07(object):
    def __init__(self, data):
        self.cf = [1, 4, 1, 1, 1, 1]
        self.cf_a = hexlist2(self.cf)
        data = data[2:]
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [
            "最高报警等级",
            "通用报警标志",
            "可充电储能装置故障总数N1",
            "驱动电机故障总数N2",
            "发动机故障总数N3",
            "其他故障总数N4",
        ]
        self.oj = list2dict(self.o, self.list_o, self.cf_a)
        self.oj2 = {'极值数据':self.oj}
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            '最高报警等级' : hex2dec(self.oj['最高报警等级'], e=True),
            '通用报警标志' : fun_07_02_07.fun_02(self.oj['通用报警标志']),
            '可充电储能装置故障总数N1' : hex2dec(self.oj['可充电储能装置故障总数N1'], e=True),
            '驱动电机故障总数N2' : hex2dec(self.oj['驱动电机故障总数N2'], e=True),
            '发动机故障总数N3' : hex2dec(self.oj['发动机故障总数N3'], e=True),
            '其他故障总数N4' : hex2dec(self.oj['其他故障总数N4'], e=True),
        }
        self.pj2 = {'极值数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]

        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_07', self.o)
        
        print('fun_07_02_07 done!')
        print(self.o)
        
    def fun_02(data):
        n = '{:032b}'.format(int(data, 16))

        baojing_list = [
            "温度差异报警",
            "电池高温报警",
            "车载储能装置类型过压报警",
            "车载储能装置类型欠压报警",
            "SOC低报警",
            "单体电池过压报警",
            "单体电池欠压报警",
            "SOC过高报警",
            "SOC跳变报警",
            "可充电储能系统不匹配报警",
            "电池单体一致性差报警",
            "绝缘报警",
            "DC-DC温度报警",
            "制动系统报警",
            "DC-DC状态报警",
            "驱动电机控制器温度报警",
            "高压互锁状态报警",
            "驱动电机温度报警",
            "车载储能装置类型过充",
        ]

        baojing = [n]

        for i in range(0,19):
            if n[-i] == "1":
                baojing.append(baojing_list[i])

        return baojing

## fun_07_02_08

In [314]:
class fun_07_02_08(object):
    def __init__(self, data):
        data = data[2:]
        self.o = data
        self.dj_n_o = data[0:2]
        self.dj_n_j = hex2dec(self.dj_n_o) # 电池个数
        
        self.cf_u = [1, 1]
        self.cf = fun_07_02_08.fun_cf(self)
        self.cf_a = hexlist2(self.cf)
        self.o = data[0:self.cf_a[-1]]
        self.list_o = [            
            "可充电储能子系统个数_电压",
            "可充电储能子系统号_电压",
            "可充电储能装置电压",
            "可充电储能装置电流",
            "单体电池总数",
            "本帧起始电池序号",
            "本帧单体电池总数",
            "单体电池电压",
        ]
        
        self.oj = fun_07_02_08.fun_oj(self)
        self.oj2 = {'可充电储能装置电压数据':self.oj}  
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "可充电储能子系统个数_电压":self.dj_n_j,
            "可充电储能子系统号_电压":[hex2dec(i) for i in self.oj['可充电储能子系统号_电压']],
            "可充电储能装置电压":[hex2dec(i, k=0.1) for i in self.oj['可充电储能装置电压']],
            "可充电储能装置电流":[hex2dec(i, k=0.1, n=-1000) for i in self.oj['可充电储能装置电流']],
            "单体电池总数":[hex2dec(i) for i in self.oj['单体电池总数']],
            "本帧起始电池序号":[hex2dec(i) for i in self.oj['本帧起始电池序号']],
            "本帧单体电池总数":[hex2dec(i) for i in self.oj['本帧单体电池总数']],
            "单体电池电压":[hex2list(i, num=2, kk=0.01) for i in self.oj['单体电池电压']],    
        }
        self.pj2 = {'可充电储能装置电压数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_07', self.o)
        
        print('fun_07_02_08 done!')
        print(self.o)
        
    
    def fun_cf(self):
        cf_u=[1,2,2,2,2,1,None]
        self.c = [] 
        data = self.o
        for i in range(self.dj_n_j):
            cf_u[6] = hex2dec(data[20:22]) * 2
            self.c = self.c + cf_u
            
            cf_a = hexlist2(self.c)
            data = data[cf_a[-1]:]
            
        self.c = [1] + self.c
            
        return self.c
    
    
    def fun_oj(self):
        data = self.o[2:]
        cf_a = hexlist2(self.cf[1:])
        dict_oj = {
            "可充电储能子系统个数_电压":self.dj_n_o,
        }
        
        list_o = [
            "可充电储能子系统号_电压",
            "可充电储能装置电压",
            "可充电储能装置电流",
            "单体电池总数",
            "本帧起始电池序号",
            "本帧单体电池总数",
            "单体电池电压",
        ]
        
        dict_oj_u = {
            "可充电储能子系统号_电压":[],
            "可充电储能装置电压":[],
            "可充电储能装置电流":[],
            "单体电池总数":[],
            "本帧起始电池序号":[],
            "本帧单体电池总数":[],
            "单体电池电压":[],
        }
        
        for i in range(self.dj_n_j):
            for j in range(len(dict_oj_u)):
                data_unit = data[cf_a[i * len(dict_oj_u) + j]:cf_a[i * len(dict_oj_u) + j +1]]
                dict_oj_u[list_o[j]].append(data_unit)
        
        dict_all = dict(dict_oj, **dict_oj_u)
        return dict_all
    

## fun_07_02_09

In [315]:
class fun_07_02_09(object):
    def __init__(self, data):
        data = data[2:]
        self.o = data
        self.dj_n_o = data[0:2]
        self.dj_n_j = hex2dec(self.dj_n_o) # 电池个数
        
        self.cf_u = [1, 1]
        self.cf = fun_07_02_09.fun_cf(self)
        self.cf_a = hexlist2(self.cf)
        
        self.o = data[0:self.cf_a[-1]]
        
        self.list_o = [
            "可充电储能子系统个数_温度",
            "可充电储能子系统号_温度",
            "可充电储能温度探针个数",
            "可充电储能子系统各温度探针检测到的温度值",
        ]
        
        self.oj = fun_07_02_09.fun_oj(self)
        self.oj2 = {'可充电储能装置温度数据':self.oj}  
        self.ol = pd.DataFrame([self.oj]).reindex(columns=self.list_o)
        self.pj = {
            "可充电储能子系统个数_温度":self.dj_n_j,
            "可充电储能子系统号_温度":[hex2dec(i) for i in self.oj['可充电储能子系统号_温度']],
            "可充电储能温度探针个数":[hex2dec(i) for i in self.oj['可充电储能温度探针个数']],
            "可充电储能子系统各温度探针检测到的温度值":[hex2list(i, num=1, kn=-40) for i in self.oj['可充电储能子系统各温度探针检测到的温度值']],
        }
        self.pj2 = {'可充电储能装置温度数据':self.pj}
        self.pl = pd.DataFrame([self.pj]).reindex(columns=self.list_o)
        
        self.next = data[len(self.o):]
        self.nextMark = data[len(self.o):len(self.o)+2]
        
        glv.set_value('data_f', self.next)
        glv.set_value('m_07_02', self.nextMark)
        
        glv.set_value('data_07_02_07', self.o)
        
        print('fun_07_02_09 done!')
        print(self.o)
        
    
    def fun_cf(self):
        cf_u=[1,2, None]
        self.c = [] 
        data = self.o
        for i in range(self.dj_n_j):
            n_str = data[4:8]
            n = hex2dec(n_str)
            cf_u[2] = n
            self.c = self.c + cf_u
            
            cf_a = hexlist2(self.c)
            data = data[cf_a[-1]:]
            
        self.c = [1] + self.c
            
        return self.c
    
    
    def fun_oj(self):
        data = self.o[2:]
        cf_a = hexlist2(self.cf[1:])
        dict_oj = {
            "可充电储能子系统个数_温度":self.dj_n_o,
        }
        
        list_o = [
            "可充电储能子系统号_温度",
            "可充电储能温度探针个数",
            "可充电储能子系统各温度探针检测到的温度值",
        ]
        
        dict_oj_u = {
            "可充电储能子系统号_温度":[],
            "可充电储能温度探针个数":[],
            "可充电储能子系统各温度探针检测到的温度值":[],
        }
        
        for i in range(self.dj_n_j):
            for j in range(len(dict_oj_u)):
                data_unit = data[cf_a[i * len(dict_oj_u) + j]:cf_a[i * len(dict_oj_u) + j +1]]
                dict_oj_u[list_o[j]].append(data_unit)
        
        dict_all = dict(dict_oj, **dict_oj_u)
        return dict_all
    

## fun_07_cursor

In [316]:
class fun_07_cursor:
    def __init__(self, data):
        
        if len(data) > 0:
            self.o = data
        else :
            self.o = None
            
        self.oj = self.pj = {'自定义': self.o}
        self.ol = self.pl = pd.DataFrame({'自定义': [self.o]})
        
        print('fun_07_curos done!')

## fun_08

In [317]:
class fun_08:
    def __init__(self):
        self.o = glv.get_value('bcc')
        
        self.oj = self.pj = {'校验码':self.o}
        self.ol = self.pl = pd.DataFrame({'校验码':[self.o]})
        
        print('fun_08 done!')

## 主体

In [318]:
class gb32960:
    
    glv._init()
    
    def __init__(self, data, mo_list=['01','02','05','06','07','08','09']):
        
        
        glv.set_value('data_mo_list', mo_list)
        
        data = data.upper()
        self.bcc = data[-2:]
        glv.set_value('bcc', self.bcc)
        data = data[:-2]
        glv.set_value('data_f', data)
        
        self.o = data
        self.f_01 = fun_01to06(glv.get_value('data_f'))
        self.f_07 = fun_07(glv.get_value('data_f'))
        self.f_08 = fun_08()
        
        self.oj = dict(self.f_01.oj, **self.f_07.oj)
        self.oj = dict(self.oj, **self.f_08.oj)
        
        self.ol = pd.merge(self.f_01.ol, self.f_07.ol, left_index=True, right_index=True)
        self.ol = pd.merge(self.ol, self.f_08.ol, left_index=True, right_index=True)
        
        
        self.pj = dict(self.f_01.pj, **self.f_07.pj)
        self.pj = dict(self.pj, **self.f_08.pj)
        
        self.pl = pd.merge(self.f_01.pl, self.f_07.pl, left_index=True, right_index=True)
        self.pl = pd.merge(self.pl, self.f_08.pl, left_index=True, right_index=True)
        
        print("all done!")

## 参数说明



全局变量：
- data_f: 剩余报文
- m_07_02: 报文标记
- bcc: 效验码
- data_mo：执行标记
- data_mo_list:

## 执行

In [321]:
filepath = r'导出报文记录-20200706093025N.csv'
mo_list = ['01','02','05','06','07','08','09']

df_o = pd.read_csv(filepath, encoding='gbk')
df_o = df_o.rename(columns={'原始报文\t': '报文内容', '服务器接收时间\t': '接收时间'})

df_o = df_o[:100]
df_o['报文内容'] = df_o['报文内容'].map(lambda p : p.strip('\t'))

In [323]:
import time

time_start=time.time()
decoded_list = []

def get_decoded_list(row):
    try:
        decoded_list.append(gb32960(row['报文内容']).pl.to_dict(orient='list'))
    except Exception as e:
        print(e)
df_o.apply(get_decoded_list, axis=1)
decoded_dataframe = pd.DataFrame(decoded_list)
time_end=time.time()
print('totally cost',time_end-time_start)
decoded_dataframe

020101044C4E204E205F0DE82710050006CF7FB401D49F560601590EAA013E0E5201154101073E070000000000000000000801010DD9233C00600001600E810E810E800E7F0E960E810E7B0E7E0E9B0E810E810E800E580E580E590E590E9E0E880E880E870E5A0E590E5A0E590E830E880E880E850E920E850E860E860E9C0E880E870E850E7F0E860E860E850E550E560E560E560E540E560E550E560E540E560E560E550E570E560E550E550E530E550E550E530E560E520E590E590E590E570E570E570E570E580E570E580E560E550E570E560E580E580E550E570E580E560E580E560E820E860E840E840EAA0E870E870E870E9B0E850E880E8809010100303F3F3F3F3F3F3E3E3F3F3F3F4040404040403F40414141414141404140404040404040403F403F403E3E3F3F40403F40 fun0201 test data_f
fun_07_02_01 done!
0201010000000DEFD60DD9233C34010F07D00000
fun_07_02_02 done!
0101044C4E204E205F0DE82710
fun_07_02_05 done!
0006CF7FB401D49F56
fun_07_02_06 done!
01590EAA013E0E5201154101073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DD9233C00600001600E810E810E800E7F0E960E810E7B0E7E0E9B0E810E810E800E580E580E590E590E9E0E880E880E870E5A0E590E5A0E59

020101014B4E204E209F0DAC2710050006CF7F7F01D49F3106010F0E5601070E4D01184201073E070000000000000000000801010DC0272E00600001600E520E550E530E530E530E540E4D0E520E530E540E520E530E550E550E560E550E550E560E560E540E560E560E560E560E530E550E540E540E530E540E550E550E540E550E550E540E510E540E530E540E530E550E540E550E530E540E530E550E530E540E540E540E550E550E530E540E510E530E540E520E520E4E0E540E550E520E530E530E530E520E530E530E550E510E510E530E510E520E540E510E530E520E520E530E520E520E530E530E530E550E550E540E530E520E520E520E530901010030403F3F403F3F3E3E3F403F4040404040414140404141414241414141404040404040404040403F403F3F3F3F40404040 fun0201 test data_f
fun_07_02_01 done!
0103010000000DEFD60DC0272E33011007D00002
fun_07_02_02 done!
0101014B4E204E209F0DAC2710
fun_07_02_05 done!
0006CF7F7F01D49F31
fun_07_02_06 done!
010F0E5601070E4D01184201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DC0272E00600001600E520E550E530E530E530E540E4D0E520E530E540E520E530E550E550E560E550E550E560E560E540E560E560E560E56

fun_07_02_08 done!
01010DC0271000600001600E550E550E530E530E530E540E4D0E520E530E540E520E530E530E520E530E530E560E550E560E550E540E540E540E530E520E550E550E540E530E530E550E550E550E550E550E540E4F0E520E500E530E4F0E520E520E520E4E0E510E500E510E520E520E530E520E540E530E520E520E500E520E520E510E520E4D0E540E540E520E520E520E540E530E540E520E550E520E510E540E510E520E540E500E530E520E530E540E520E530E520E530E530E550E550E540E530E520E520E520E53
fun_07_02_09 done!
0101003040403F403F3F3E3E3F403F40404040404141404041414142414241414040404040404040404040403F3F3F3F41414040
fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
020101014B4E204E20A00DAC2710050006CF7F7C01D49F310601130E5601070E4D01184201073E070000000000000000000801010DBF271000600001600E530E540E520E520E540E540E4D0E530E530E540E530E530E530E520E530E530E550E550E560E550E540E540E540E530E4E0E510E500E4F0E500E510E510E520E520E520E520E520E4F0E520E500E530E4F0E520E520E520E4E0E510E500E510E520E520E530E520E540E530E520E520E500E520E520E510E520E4D0E540E540

fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
020101024764D24D26A20DB62597050006CF8B8D01D49E860601350E6601510DC201164201073E070000000000000000000801010DAF256200600001600E3E0E3E0E3B0E3B0E390E3E0E370E3C0E390E3D0E3C0E3C0E560E550E560E550E3B0E430E430E410E610E5D0E5E0E5D0E590E5D0E5C0E5A0E5D0E5B0E5C0E5B0E610E5D0E5C0E5A0E570E5B0E5A0E5C0E600E5A0E5B0E5A0E530E540E530E540E5C0E5D0E5E0E5D0E660E5D0E5E0E5D0E620E5D0E5D0E5A0E5A0E570E5C0E5C0E450E490E4B0E4B0DE80DFA0DFA0DFD0DF90DE80DF10DEC0DDE0DF10DEE0DF30DC20DEE0DF10DF20E430E430E450E440E390E450E440E430E660E5F0E600E61090101003040403F403F3F3E3F3F404040404040404141404041424142414241414040404040404040404040403F3F3F3F41414040 fun0201 test data_f
fun_07_02_01 done!
01030102DD000DEFD30DAF256233010E07D00000
fun_07_02_02 done!
0101024764D24D26A20DB62597
fun_07_02_05 done!
0006CF8B8D01D49E86
fun_07_02_06 done!
01350E6601510DC201164201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DAF256200600001600E3E0E3E0E3B0E3B0E390E3E0

fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
02010101485EC44F10A10D982873050006CFCA1201D48B610601150E6101390DD401184201073E070000000000000000000801010DB728E600600001600E590E5B0E590E580E5A0E5B0E530E590E5A0E5A0E5A0E5A0E5C0E5C0E5E0E5D0E5A0E5C0E5C0E5B0E610E5D0E5E0E5D0E430E440E440E420E5B0E5C0E5C0E5D0E5F0E5E0E5D0E5C0E560E580E570E590E140E2F0E2E0E300E1E0E2F0E2C0E310E5F0E600E610E610DE40E0A0E080E0B0DD40E030E030E080E4B0E470E4C0E4C0E480E4B0E4C0E4D0E490E4B0E4B0E4D0E4B0E490E4B0E480E490E4C0E490E4C0E460E4B0E4C0E4B0E590E5A0E5A0E5A0E5B0E5B0E5B0E5A0E580E590E590E5A09010100303F403F403F3F3E3F3F403F4040404040414040404141414241414141404040404040404040403F403F3F3F3F40404040 fun0201 test data_f
fun_07_02_01 done!
0103010218000DEFBE0DB728E634012E07D01400
fun_07_02_02 done!
010101485EC44F10A10D982873
fun_07_02_05 done!
0006CFCA1201D48B61
fun_07_02_06 done!
01150E6101390DD401184201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DB728E600600001600E590E5B0E590E580E5A0E5B0

...

fun_07_02_09 done!
0101003040403F403F3F3F3F3F404040404040414141404041424142414241414041404140404040404140403F3F3F3F41414040
fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
020101014A4E204E20A10DC02710050006CFDDC901D480210601510E6B01590D9D01164201033F070000000000000000000801010DB8271A00600001600E540E560E540E540E110E290E210E290E090E270E260E260E630E630E630E620E530E550E550E540E640E620E620E610E540E560E570E560E540E560E560E560E580E590E590E580E540E570E550E570E560E580E570E570E560E570E570E580E570E580E580E570E580E580E570E570E550E570E570E550E570E510E580E580E580E570E570E580E580E5A0E5A0E5A0E590E560E5A0E570E4F0E520E4E0E510E6B0E620E640E610DF50DED0DF10DF00D9D0DF20DF10DF10E670E5E0E600E60090101003040403F403F3F3F3F3F404040404040414141404041424142424241414041404140404040404140403F3F3F3F41414040 fun0201 test data_f
fun_07_02_01 done!
0103010000000DEFBB0DB8271A35011007D00006
fun_07_02_02 done!
0101014A4E204E20A10DC02710
fun_07_02_05 done!
0006CFDDC901D48021
fun_07_02_06 done!
01510E6B0

fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
02010102455B684D1CA00DC02616050006CFFB3301D470AB06015D0E6C01560E3701164201073E070000000000000000000801010DC7261600600001600E5C0E5E0E5D0E5C0E390E450E3D0E440E4B0E4E0E4F0E4E0E490E460E460E470E510E540E540E520E620E610E610E610E630E660E660E640E660E640E650E660E6A0E670E660E640E610E640E630E650E6A0E650E650E650E670E650E640E640E630E650E650E640E600E5F0E5E0E5E0E570E590E5A0E580E5F0E5A0E620E610E5C0E5B0E5C0E5D0E640E630E620E640E600E610E630E610E5B0E5C0E580E5B0E570E570E580E570E390E370E390E390E560E570E570E560E6C0E650E660E66090101003040403F403F3F3E3F3F404040404040414141404041424142414241414040404140404040404040403F3F3F3F41414040 fun0201 test data_f
fun_07_02_01 done!
01030101B1000DEFB20DC7261635010E07D00000
fun_07_02_02 done!
010102455B684D1CA00DC02616
fun_07_02_05 done!
0006CFFB3301D470AB
fun_07_02_06 done!
015D0E6C01560E3701164201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DC7261600600001600E5C0E5E0E5D0E5C0E390E450

fun_07 done!
fun_08 done!
all done!
020101024462F24D1CA00DC025C3050006D00E4101D46B4F0601050E6A01390DCD01164201073E070000000000000000000801010DB4259400600001600E670E670E650E640E6A0E670E5E0E650E650E630E630E640E4C0E4C0E4C0E4C0E610E610E620E600E4B0E4C0E4E0E4C0E600E620E630E600E600E610E610E610E610E610E610E610E5D0E600E5F0E610E5F0E610E610E610E600E610E600E610E080E040E050E060DDC0E030E020E050DCD0DFC0DFD0E030E080DFA0E030E080DD40E020E020E070DF00E020E020E060E620E630E650E610E680E680E630E660E660E650E650E630E610E610E620E620E610E610E600E600E5D0E5E0E5F0E6009010100303F403F403F3F3E3F3F404040404040414141404041424142414241414040404040404040404040403F3F3F3F41414040 fun0201 test data_f
fun_07_02_01 done!
010301029E000DEFAA0DB4259436010E07D00000
fun_07_02_02 done!
0101024462F24D1CA00DC025C3
fun_07_02_05 done!
0006D00E4101D46B4F
fun_07_02_06 done!
01050E6A01390DCD01164201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DB4259400600001600E670E670E650E640E6A0E670E5E0E650E650E630E630E640E4C0E4C0E4C

fun_07_02_06 done!
01210E7301110E1501184201073E
fun_07_02_07 done!
000000000000000000
fun_07_02_08 done!
01010DC9260C00600001600E610E630E610E600E540E5B0E520E590E3A0E480E480E480E640E620E630E650E150E300E310E2E0E5B0E5E0E5D0E5D0E540E560E560E550E6F0E6B0E6C0E6B0E730E6E0E6D0E6C0E5B0E5E0E5D0E5F0E730E6C0E6C0E6C0E700E6C0E6C0E6C0E6B0E6C0E6C0E6B0E720E6C0E6C0E6B0E700E6C0E6C0E6A0E6A0E660E6D0E6C0E610E610E620E620E5D0E5F0E5F0E610E5F0E5D0E600E5D0E360E400E3B0E400E470E4F0E500E4E0E4F0E4F0E500E4F0E570E5A0E590E590E540E560E570E57
fun_07_02_09 done!
010100303F403F403F3F3E3F3F404040404040404141404041414142414241414040404040404040404040403F3F3F3F41414040
fun_07_02 done!
fun_07_curos done!
fun_07 done!
fun_08 done!
all done!
02010101485EB053529F0D5C2D51050006CFE38501D47C480601100E6501110E1501184201073E070000000000000000000801010DC22E3600600001600E610E630E610E600E540E5B0E520E590E3A0E480E480E480E640E620E630E650E150E300E310E2E0E5B0E5E0E5D0E5D0E540E560E560E550E500E540E550E560E4F0E550E550E550E520E530E520E550E4B0E540E5

,起始符,命令标识,应答标志,唯一识别码,数据单元加密方式,数据单元长度,01,车辆状态,充电状态,运行模式,...,本帧单体电池总数,单体电池电压,可充电储能子系统个数_温度,可充电储能子系统号_温度,可充电储能温度探针个数,可充电储能子系统各温度探针检测到的温度值,自定义,校验码,登出时间,登出流水号
0,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[熄火],[停车充电],[纯电],...,[[96]],"[[[37.13, 37.13, 37.12, 37.11, 37.34, 37.13, 3...",[1],[[1]],[[48]],"[[[23, 23, 23, 23, 23, 23, 22, 22, 23, 23, 23,...",[None],[75],NaN,NaN
1,[##],[车辆登出],[命令],[L0000000000000001],[不加密],[8],NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,[None],[C8],[2019-04-29 23:56:39],[10]
2,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[熄火],[未充电],[纯电],...,[[96]],"[[[36.66, 36.69, 36.66, 36.66, 36.68, 36.69, 3...",[1],[[1]],[[48]],"[[[23, 23, 23, 24, 23, 23, 22, 22, 23, 23, 23,...",[None],[80],NaN,NaN
3,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[熄火],[未充电],[纯电],...,[[96]],"[[[36.67, 36.69, 36.66, 36.66, 36.68, 36.69, 3...",[1],[[1]],[[48]],"[[[23, 23, 23, 24, 23, 23, 22, 22, 23, 23, 23,...",[None],[86],NaN,NaN
4,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[熄火],[未充电],[纯电],...,[[96]],"[[[36.67, 36.69, 36.66, 36.66, 36.68, 36.69, 3...",[1],[[1]],[[48]],"[[[24, 23, 23, 24, 23, 23, 22, 22, 23, 24, 23,...",[None],[C9],NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[启动],[未充电],[纯电],...,[[96]],"[[[36.81, 36.83, 36.81, 36.8, 36.68, 36.75, 36...",[1],[[1]],[[48]],"[[[23, 24, 23, 24, 23, 23, 22, 23, 23, 24, 24,...",[None],[3C],NaN,NaN
95,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[启动],[未充电],[纯电],...,[[96]],"[[[36.84, 36.84, 36.83, 36.83, 36.84, 36.85, 3...",[1],[[1]],[[48]],"[[[24, 24, 23, 24, 23, 23, 22, 23, 23, 24, 24,...",[None],[70],NaN,NaN
96,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[启动],[未充电],[纯电],...,[[96]],"[[[36.79, 36.79, 36.77, 36.77, 37.0, 36.93, 36...",[1],[[1]],[[48]],"[[[24, 24, 23, 24, 23, 23, 22, 23, 23, 24, 24,...",[None],[CC],NaN,NaN
97,[##],[实时信息上报],[命令],[L0000000000000001],[不加密],[333],[01],[启动],[未充电],[纯电],...,[[96]],"[[[36.9, 36.94, 36.92, 36.91, 37.0, 36.93, 36....",[1],[[1]],[[48]],"[[[23, 24, 23, 24, 23, 23, 23, 23, 24, 24, 24,...",[None],[E4],NaN,NaN


In [285]:
decoded_dataframe.to_excel('D:/decoded_files/decoded_data.xlsx')